# Advanced Topics in Stochastic Programming

This notebook contains advanced topics for optimization under uncertainty.

```{warning}
**AI-drafted prose, not yet reviewed by Prof. Dowling.**

Some of the writing on this page was drafted or edited by an AI assistant and has not yet been reviewed: 5 rewritten markdown cells, measured against the last version of this notebook predating AI editing (2026-08-17).

This notice is about the *prose only*. It says nothing either way about the code, the numbers or the figures, which are checked separately. It is removed once the page has been reviewed.
```

## Sample average approximation

Sample average approximation (SAA) replaces an expectation with a finite sample:

\[
z_N = \min_{\mathbf{x}\in X}\frac{1}{N}\sum_{i=1}^{N}g(\mathbf{x},\boldsymbol{\xi}^i),
\qquad \boldsymbol{\xi}^i \overset{\mathrm{iid}}{\sim} P.
\]

A responsible SAA study separates:

* a **training sample** used to choose \(x_N\);
* a larger, independent **evaluation sample** used to estimate out-of-sample performance; and
* repeated samples used to measure solution variability.

One SAA solve is not a statistical guarantee. Sample size, random seed, and evaluation protocol are part of the model.
A fixed candidate must be independent of assessment data. Within an assessment batch, candidate evaluation and re-optimization may deliberately share common random numbers; the paired gap is nonnegative. Independent replications estimate its uncertainty. See Mak, Morton and Wood (1999), and Bayraksan and Morton (2006).


## Quadrature and sparse grids

Quadrature replaces an integral with a weighted sum,

\[
\int f(x)\,dx \approx \sum_{i=1}^{n}\omega_i f(x_i).
\]

Tensor-product quadrature grows exponentially with dimension. Sparse grids retain selected tensor-product points and can be effective for smooth, moderately dimensional integrands. They do not universally "defeat" dimensionality: performance depends on dimension, regularity, anisotropy, and the grid rule.

For \(y(x)=7x^3-8x^2-3x+3\) on \([-1,1]\):

* the exact integral is \(2/3\);
* the endpoint trapezoid gives \(-10\); and
* two-point Gauss--Legendre quadrature is exact because the integrand is cubic.

![Exact cubic together with the two-point trapezoidal and Gauss--Legendre interpolants.](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/quadrature-gauss-vs-trapezoid.png)

In [1]:
# Install course dependencies before importing Pyomo on Colab.
import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper

helper.set_plotting_style()

import numpy as np

lower, upper = -1.0, 1.0


def integrand(x):
    return 7 * x**3 - 8 * x**2 - 3 * x + 3


# Odd powers integrate to zero; integrate -8*x**2 + 3 analytically.
exact = 2 / 3
trapezoid = (upper - lower) * (integrand(lower) + integrand(upper)) / 2
# Two nodes integrate polynomials through degree 3 exactly on [-1, 1].
nodes, weights = np.polynomial.legendre.leggauss(2)
gauss_legendre = sum(weights * integrand(nodes))

print(f"exact: {exact:.12f}")
print(f"trapezoid: {trapezoid:.12f}")
print(f"two-point Gauss--Legendre: {gauss_legendre:.12f}")
assert abs(gauss_legendre - exact) < 1e-12

exact: 0.666666666667
trapezoid: -10.000000000000
two-point Gauss--Legendre: 0.666666666667


### Python implementations

NumPy supplies one-dimensional Gauss--Legendre nodes through **numpy.polynomial.legendre.leggauss**. The optional [Tasmanian](https://ornl.github.io/TASMANIAN/rolling/) package provides several sparse-grid constructions.

The course does not require Tasmanian. This notebook keeps the executable calculation dependency-light and uses the repository figure below for the sparse-grid concept.

The comparison uses the same number of points in two dimensions:

* the sparse grid is a union of tensor products of nested Clenshaw--Curtis nodes; and
* the Monte Carlo sample uses a fixed random seed.

Visual coverage alone does not establish accuracy. Compare integration error on an independent test problem.

![Nested two-dimensional sparse-grid points beside the same number of seeded Monte Carlo samples.](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/sparse-grid-vs-monte-carlo.png)

## References

* Mak, Morton and Wood (1999), *Monte Carlo bounding techniques for determining solution quality in stochastic programs*, DOI: [10.1016/S0167-6377(98)00054-6](https://doi.org/10.1016/S0167-6377(98)00054-6).
* Bayraksan and Morton (2006), *Assessing solution quality in stochastic programs*, DOI: [10.1007/s10107-006-0720-x](https://doi.org/10.1007/s10107-006-0720-x).
* [NumPy Gauss–Legendre documentation](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.legendre.leggauss.html): exactness through degree $2n-1$.
* [Tasmanian project documentation](https://ornl.github.io/TASMANIAN/rolling/): optional sparse-grid software.

The two comparison figures are original course illustrations. The sparse-grid picture displays nodes, not quadrature weights or an error bound.
